# A logistic-regression risk score

Exercise notebook for the lesson **[A logistic-regression risk score](https://ramkumarmd.com/medical-ai/machine-learning/logistic-regression-risk-score)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Logistic regression models the log-odds of an outcome as a weighted sum of predictors, giving a probability for each patient and an odds ratio for each predictor. For many clinical prediction problems with tabular data it performs as well as more complex machine-learning methods, is easy to check and explain, and can be turned into a simple points score. Building one well means choosing predictors in advance, having enough outcome events, and checking discrimination and calibration on held-out data.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic cohort

Synthetic adults with a known relationship between predictors and five-year diabetes risk.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(5)
n = 5000
df = pd.DataFrame({
    "age": rng.integers(30, 75, n),
    "bmi": rng.normal(27, 4.5, n).round(1),
    "fasting_glucose": rng.normal(100, 10, n).round(),   # mg/dL
    "family_history": rng.binomial(1, 0.3, n),
    "hypertension": rng.binomial(1, 0.35, n),
})
logit = (-17.5 + 0.03 * df.age + 0.10 * df.bmi + 0.11 * df.fasting_glucose
         + 0.6 * df.family_history + 0.4 * df.hypertension)
df["diabetes_5y"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"{n} synthetic adults, 5-year incidence {df.diabetes_5y.mean():.1%}")

# 5000 synthetic adults, 5-year incidence 19.4%

### Fitting the model and reading odds ratios

Continuing from the previous block: hold out 30% for testing, fit, and show odds ratios.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

predictors = ["age", "bmi", "fasting_glucose", "family_history", "hypertension"]
train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.diabetes_5y)

model = LogisticRegression(penalty=None, max_iter=1000)
model.fit(train[predictors], train.diabetes_5y)
odds_ratios = pd.Series(np.exp(model.coef_[0]), index=predictors)
print(odds_ratios.round(2))

# age                1.03
# bmi                1.09
# fasting_glucose    1.12
# family_history     2.04
# hypertension       1.65

### Checking discrimination and calibration

Continuing from the previous block: AUROC and calibration on the held-out 30%.

In [ ]:
from sklearn.metrics import roc_auc_score

p = model.predict_proba(test[predictors])[:, 1]
print(f"Test AUROC {roc_auc_score(test.diabetes_5y, p):.3f}")

# Calibration slope and intercept: regress the outcome on the model's log-odds
log_odds = np.log(p / (1 - p)).reshape(-1, 1)
recal = LogisticRegression(penalty=None).fit(log_odds, test.diabetes_5y)
print(f"Calibration slope {recal.coef_[0][0]:.2f}, intercept {recal.intercept_[0]:.2f}")
print(f"Mean predicted risk {p.mean():.1%}, observed {test.diabetes_5y.mean():.1%}")

# Test AUROC 0.793
# Calibration slope 0.99, intercept 0.04
# Mean predicted risk 18.7%, observed 19.4%

### From model to points score

Continuing from the previous block: integer points, then the AUROC of the points score.

In [ ]:
coef = pd.Series(model.coef_[0], index=predictors)
steps = {"age": 10, "bmi": 5, "fasting_glucose": 10, "family_history": 1, "hypertension": 1}
unit = coef["age"] * 10
points = {k: round(coef[k] * s / unit) for k, s in steps.items()}
print(points)

score = ((test.age - 30) / 10 * points["age"]
         + (test.bmi - 25) / 5 * points["bmi"]
         + (test.fasting_glucose - 90) / 10 * points["fasting_glucose"]
         + test.family_history * points["family_history"]
         + test.hypertension * points["hypertension"])
print(f"Points score AUROC {roc_auc_score(test.diabetes_5y, score):.3f}")

# {'age': 1, 'bmi': 2, 'fasting_glucose': 4, 'family_history': 2, 'hypertension': 2}
# Points score AUROC 0.793

## Exercise

**Goal:** Develop, evaluate and simplify a logistic-regression risk model on synthetic data, and test how it holds up when the population changes.

**Data:** Synthetic adults generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the model, odds ratios, test AUROC and calibration slope and intercept.

In [ ]:
# Your code here


### Step 2

Draw a calibration plot with 10 risk groups and a decision curve for thresholds from 5% to 40%.

In [ ]:
# Your code here


### Step 3

Simulate a 'new population' with higher average BMI and a different baseline risk, then measure AUROC and calibration there. Recalibrate the intercept and compare.

In [ ]:
# Your code here


### Step 4

Build the points score, map each total to observed risk in the test set, and present it as a small lookup table.

In [ ]:
# Your code here


### Step 5

Fit an L2-penalised version and a gradient-boosted model, and compare their AUROC and calibration with the unpenalised model.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/logistic-regression-risk-score](https://ramkumarmd.com/medical-ai/machine-learning/logistic-regression-risk-score). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.